# Contextual Bandit — Smart Scheduler

This notebook develops the online scanner/scheduler.

The scheduler operates under the real receiver constraint:

- 36 frequency bands
- one band can be scanned per 50 ms time slot
- the scheduler does NOT have access to future RF ground truth
- the scheduler only uses information available up to the current timestep

Current development stages:

1. Build a reusable online scheduling simulator.
2. Build a smart heuristic scheduler.
3. Compare it against simple baselines.
4. Add contextual bandit learning.
5. Add a neural contextual bandit.

The Oracle is implemented separately in `Oracle.ipynb`.
It has access to the complete RF environment and provides a
ground-truth performance reference.

Primary evaluation objective:
    maximize the number of transmission events intercepted.

Secondary evaluation objective:
    minimize interception delay among intercepted events.

In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
from collections import defaultdict, deque
import random
import json

import numpy as np
import matplotlib.pyplot as plt

print("NumPy:", np.__version__)

NumPy: 2.2.6


In [2]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# RF environment directory
# ------------------------------------------------------------

TRAIN_RFENV_DIR = Path("./train_RFEnvs")

# ------------------------------------------------------------
# RF representation
# ------------------------------------------------------------

FREQ_MIN_MHZ = 0
FREQ_MAX_MHZ = 18_000
FREQ_BIN_MHZ = 500

TIME_BIN_S = 0.050

N_FREQ_BINS = int(
    (FREQ_MAX_MHZ - FREQ_MIN_MHZ)
    / FREQ_BIN_MHZ
)

assert N_FREQ_BINS == 36

print("Frequency bands :", N_FREQ_BINS)
print("Frequency bin   :", FREQ_BIN_MHZ, "MHz")
print("Time resolution :", TIME_BIN_S * 1000, "ms")

Frequency bands : 36
Frequency bin   : 500 MHz
Time resolution : 50.0 ms


In [3]:
# ============================================================
# 2. LOAD RF ENVIRONMENTS
# ============================================================

rf_files = sorted(
    p for p in TRAIN_RFENV_DIR.glob("*.npy")
    if p.name != "environment_config.npy"
)

if len(rf_files) == 0:
    raise FileNotFoundError(
        f"No RF environment files found in "
        f"{TRAIN_RFENV_DIR.resolve()}"
    )

print(
    f"Found {len(rf_files)} RF environments."
)

Found 5 RF environments.


## Scheduler design

At timestep t the scheduler receives a belief vector:

    P_t[b] = estimated probability that band b is transmitting.

The scheduler chooses exactly one band:

    action_t ∈ {0, 1, ..., 35}

The actual RF environment is used only by the simulator to produce
the observation after the action.

Therefore:

    belief → scheduler → action → RF observation

The scheduler never receives future ground truth.

The first scheduler will be heuristic rather than learned.

Its purpose is to establish a strong non-learning baseline before
introducing contextual-bandit learning.

In [4]:
# ============================================================
# 3. TRANSMISSION EVENT EXTRACTION
# ============================================================

def extract_transmission_events(env):
    """
    Extract continuous transmission events.

    A transmission event is a continuous run of 1s in one band.

    A 0 breaks the event into a new event.

    Returns a list of dictionaries containing:
        id
        band
        start
        end
        duration
    """

    env = np.asarray(env)

    if env.ndim != 2:
        raise ValueError(
            f"Expected 2-D environment, got {env.shape}"
        )

    if env.shape[1] != N_FREQ_BINS:
        raise ValueError(
            f"Expected {N_FREQ_BINS} bands, "
            f"got {env.shape[1]}"
        )

    events = []
    event_id = 0

    n_time_steps = env.shape[0]

    for band in range(N_FREQ_BINS):

        in_event = False
        start = None

        for t in range(n_time_steps):

            active = bool(env[t, band] > 0)

            if active and not in_event:

                start = t
                in_event = True

            elif not active and in_event:

                end = t - 1

                events.append({
                    "id": event_id,
                    "band": band,
                    "start": start,
                    "end": end,
                    "duration": end - start + 1,
                })

                event_id += 1
                in_event = False

        # Event reaches the end of the environment
        if in_event:

            end = n_time_steps - 1

            events.append({
                "id": event_id,
                "band": band,
                "start": start,
                "end": end,
                "duration": end - start + 1,
            })

            event_id += 1

    events.sort(
        key=lambda e: (e["start"], e["band"])
    )

    for new_id, event in enumerate(events):
        event["id"] = new_id

    return events

In [5]:
# ============================================================
# 4. BUILD EVENT LOOKUP
# ============================================================

def build_event_lookup(events, n_time_steps):
    """
    Build a lookup structure for online evaluation.

    active_events[t] contains the IDs of all transmission events
    that are active at timestep t.
    """

    active_events = [
        []
        for _ in range(n_time_steps)
    ]

    for event in events:

        for t in range(
            event["start"],
            event["end"] + 1
        ):

            active_events[t].append(
                event["id"]
            )

    return active_events

## Information boundary

The simulator knows:

    RF ground truth
    transmission events
    which events have already been intercepted

The scheduler does NOT receive those directly.

The simulator uses them only to evaluate the scheduler after it
selects an action.

This keeps the distinction clear:

    OBSERVATION / BELIEF
        ↓
    scheduler decision
        ↓
    ground-truth evaluation

In [6]:
# ============================================================
# 5. SMART SCHEDULER
# ============================================================

class SmartScheduler:
    """
    Non-learning heuristic scheduler.

    Uses:
        1. Current belief probability.
        2. Time since the band was last scanned.
        3. Recent observation history.

    The scheduler is intentionally interpretable.

    It does NOT access future RF truth.
    """

    def __init__(
        self,
        n_bands=N_FREQ_BINS,
        exploration_weight=0.10,
        recent_hit_penalty=0.25,
        hit_memory=5,
    ):

        self.n_bands = n_bands

        self.exploration_weight = (
            exploration_weight
        )

        self.recent_hit_penalty = (
            recent_hit_penalty
        )

        self.hit_memory = hit_memory

        self.reset()

    def reset(self):

        # Number of timesteps since each band
        # was last scanned.
        self.age_since_scan = np.full(
            self.n_bands,
            np.inf,
            dtype=np.float32
        )

        # Recent observations for each band.
        self.recent_observations = [
            deque(
                maxlen=self.hit_memory
            )
            for _ in range(self.n_bands)
        ]

    def select_action(self, belief):
        """
        Select the next band.

        Parameters
        ----------
        belief : array-like, shape (n_bands,)
            Current recursive-model belief.

        Returns
        -------
        action : int
            Selected frequency-band index.
        """

        belief = np.asarray(
            belief,
            dtype=np.float32
        )

        if belief.shape != (
            self.n_bands,
        ):
            raise ValueError(
                f"Expected belief shape "
                f"({self.n_bands},), "
                f"got {belief.shape}"
            )

        # ----------------------------------------------------
        # Base score = current probability
        # ----------------------------------------------------

        score = belief.copy()

        # ----------------------------------------------------
        # Exploration bonus
        #
        # Normalize age so the bonus stays bounded.
        # ----------------------------------------------------

        finite_ages = self.age_since_scan[
            np.isfinite(self.age_since_scan)
        ]

        if len(finite_ages) == 0:

            normalized_age = np.ones(
                self.n_bands,
                dtype=np.float32
            )

        else:

            max_age = max(
                float(np.max(finite_ages)),
                1.0
            )

            age = np.minimum(
                self.age_since_scan,
                max_age
            )

            normalized_age = (
                age / max_age
            )

            normalized_age[
                ~np.isfinite(normalized_age)
            ] = 1.0

        score += (
            self.exploration_weight
            * normalized_age
        )

        # ----------------------------------------------------
        # Recent-hit penalty
        #
        # If a band was recently observed as active,
        # repeatedly selecting it becomes less attractive.
        #
        # This helps prevent the persistent-band problem.
        # ----------------------------------------------------

        for band in range(
            self.n_bands
        ):

            history = (
                self.recent_observations[band]
            )

            if history:

                recent_hit_rate = (
                    sum(history)
                    / len(history)
                )

                score[band] -= (
                    self.recent_hit_penalty
                    * recent_hit_rate
                )

        # ----------------------------------------------------
        # Select highest score
        # ----------------------------------------------------

        action = int(
            np.argmax(score)
        )

        return action

    def update(
        self,
        action,
        observation
    ):
        """
        Update scheduler memory after receiving
        the observation from the selected band.
        """

        # Everyone gets one timestep older.
        self.age_since_scan += 1

        # Selected band was just scanned.
        self.age_since_scan[action] = 0

        # Store its latest observation.
        self.recent_observations[
            action
        ].append(
            int(observation > 0)
        )

## Important distinction

The SmartScheduler is NOT a contextual bandit yet.

It has:

    fixed hand-designed rules
    + current belief
    + observation history

There are no learned parameters.

Later, the contextual bandit will replace the fixed decision rule
with a learned policy.

The simulator and evaluation code will remain unchanged.

In [7]:
# ============================================================
# 8. GENERIC POLICY INTERFACE
# ============================================================

class Policy:

    def reset(self):
        pass

    def select_action(self, belief):
        raise NotImplementedError

    def update(
        self,
        action,
        observation,
        reward=None
    ):
        pass

In [8]:
# ============================================================
# 9. SMART SCHEDULER POLICY
# ============================================================

class SmartSchedulerPolicy(Policy):

    def __init__(
        self,
        **scheduler_kwargs
    ):

        self.scheduler = SmartScheduler(
            **scheduler_kwargs
        )

    def reset(self):

        self.scheduler.reset()

    def select_action(self, belief):

        return self.scheduler.select_action(
            belief
        )

    def update(
        self,
        action,
        observation,
        reward=None
    ):

        # SmartScheduler does not use the reward.
        # It is retained as a baseline policy.
        self.scheduler.update(
            action,
            observation
        )

In [9]:
# ============================================================
# 10. ONLINE SCHEDULING SIMULATOR
# ============================================================

def run_policy_on_environment(
    env,
    policy,
    belief_provider,
    reward_function=None,
    reset_policy=True,
):
    """
    Run an online scheduling policy against one RF environment.

    The policy sees only:
        - current belief
        - current observation
        - reward derived from observable information

    Ground-truth event information is used ONLY for evaluation.

    Parameters
    ----------
    env : np.ndarray
        Ground-truth RF environment.
        Shape: (time_steps, n_bands)

    policy : Policy
        Scheduler selecting one band per timestep.

    belief_provider : callable
        Generates the recursive belief state.

    reward_function : callable or None
        Computes reward from online-observable information.

    Returns
    -------
    result : dict
        Complete trajectory and evaluation information.
    """

    env = np.asarray(
        env,
        dtype=np.float32
    )

    n_time_steps = len(env)

    events = extract_transmission_events(
        env
    )

    active_events = build_event_lookup(
        events,
        n_time_steps
    )

    event_by_id = {
        event["id"]: event
        for event in events
    }

    # --------------------------------------------------------
    # Reset policy and reward state
    # --------------------------------------------------------

    if reset_policy:
        policy.reset()

    if hasattr(
        belief_provider,
        "reset"
    ):
        belief_provider.reset()

    if reward_function is not None:
        reward_function.reset()

    # --------------------------------------------------------
    # Evaluation state
    #
    # These structures use ground truth but are NEVER
    # passed to the policy.
    # --------------------------------------------------------

    intercepted_event_ids = set()

    interception_times = {}

    # --------------------------------------------------------
    # Online trajectory
    # --------------------------------------------------------

    actions = []
    observations = []
    beliefs = []
    rewards = []

    # --------------------------------------------------------
    # Initial belief
    # --------------------------------------------------------

    belief = belief_provider(
        t=0,
        action=None,
        observation=None
    )

    # --------------------------------------------------------
    # Main online loop
    # --------------------------------------------------------

    for t in range(
        n_time_steps
    ):

        # ----------------------------------------------------
        # 1. Scheduler chooses ONE band.
        # ----------------------------------------------------

        action = policy.select_action(
            belief
        )

        action = int(action)

        if not (
            0 <= action < N_FREQ_BINS
        ):
            raise ValueError(
                f"Invalid action {action}"
            )

        # ----------------------------------------------------
        # 2. Receiver observes ONLY selected band.
        # ----------------------------------------------------

        observation = float(
            env[t, action]
        )

        # ----------------------------------------------------
        # 3. Store online information.
        # ----------------------------------------------------

        actions.append(action)

        observations.append(
            observation
        )

        beliefs.append(
            np.asarray(
                belief,
                dtype=np.float32
            ).copy()
        )

        # ----------------------------------------------------
        # 4. Calculate reward.
        #
        # This MUST use only information available after
        # the scan.
        # ----------------------------------------------------

        if reward_function is not None:

            reward = float(
                reward_function(
                    action,
                    observation
                )
            )

        else:

            reward = 0.0

        rewards.append(
            reward
        )

        # ----------------------------------------------------
        # 5. Ground-truth evaluation ONLY.
        #
        # The policy never receives this information.
        # ----------------------------------------------------

        for event_id in active_events[t]:

            if event_id in intercepted_event_ids:
                continue

            event = event_by_id[
                event_id
            ]

            if event["band"] == action:

                intercepted_event_ids.add(
                    event_id
                )

                interception_times[
                    event_id
                ] = t

        # ----------------------------------------------------
        # 6. Update policy using observed feedback.
        # ----------------------------------------------------

        policy.update(
            action,
            observation,
            reward
        )

        # ----------------------------------------------------
        # 7. Recursive ML belief update.
        # ----------------------------------------------------

        belief = belief_provider(
            t=t + 1,
            action=action,
            observation=observation
        )

    # --------------------------------------------------------
    # Convert interception information into metrics
    # --------------------------------------------------------

    intercepted_events = []

    for event_id in intercepted_event_ids:

        event = event_by_id[
            event_id
        ]

        interception_time = (
            interception_times[event_id]
        )

        delay_steps = (
            interception_time
            - event["start"]
        )

        intercepted_events.append({

            **event,

            "interception_time":
                interception_time,

            "delay_steps":
                delay_steps,

            "delay_s":
                delay_steps * TIME_BIN_S,
        })

    missed_events = [
        event
        for event in events
        if event["id"]
        not in intercepted_event_ids
    ]

    n_events = len(events)

    n_intercepted = len(
        intercepted_events
    )

    event_interception_rate = (
        n_intercepted / n_events
        if n_events > 0
        else np.nan
    )

    if intercepted_events:

        delays = np.asarray([
            event["delay_s"]
            for event
            in intercepted_events
        ])

        mean_delay = float(
            np.mean(delays)
        )

        total_delay = float(
            np.sum(delays)
        )

    else:

        mean_delay = np.nan
        total_delay = 0.0

    unique_bands = len({
        event["band"]
        for event
        in intercepted_events
    })

    return {

        "events":
            events,

        "intercepted_events":
            intercepted_events,

        "missed_events":
            missed_events,

        "actions":
            np.asarray(actions),

        "observations":
            np.asarray(observations),

        "beliefs":
            np.asarray(beliefs),

        "rewards":
            np.asarray(rewards),

        "intercepted_event_count":
            n_intercepted,

        "total_event_count":
            n_events,

        "event_interception_rate":
            event_interception_rate,

        "mean_intercept_delay_s":
            mean_delay,

        "total_intercept_delay_s":
            total_delay,

        "unique_bands_found":
            unique_bands,
    }

Now we start with testing out actual bandit models

In [10]:
# ============================================================
# 11. LOAD TRAINED RECURSIVE MODEL
# ============================================================

import torch
import torch.nn as nn

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

Device: cuda


In [11]:
# ============================================================
# 12. RECURSIVE BELIEF UPDATE MODEL
# ============================================================

class BeliefUpdater(nn.Module):

    def __init__(
        self,
        n_bands=36,
        hidden_size=128
    ):
        super().__init__()

        self.n_bands = n_bands

        input_size = (
            n_bands      # previous probabilities
            + n_bands    # chosen action (one-hot)
            + 1          # hit / miss
        )

        self.network = nn.Sequential(
            nn.Linear(
                input_size,
                hidden_size
            ),
            nn.ReLU(),

            nn.Linear(
                hidden_size,
                hidden_size
            ),
            nn.ReLU(),

            nn.Linear(
                hidden_size,
                n_bands
            )
        )

    def forward(
        self,
        probabilities,
        actions,
        observations
    ):

        # Convert selected band to one-hot.
        action_one_hot = torch.zeros(
            actions.shape[0],
            self.n_bands,
            device=actions.device
        )

        action_one_hot.scatter_(
            1,
            actions.unsqueeze(1),
            1.0
        )

        # Convert observation to column vector.
        observations = (
            observations
            .float()
            .unsqueeze(1)
        )

        x = torch.cat(
            [
                probabilities,
                action_one_hot,
                observations
            ],
            dim=1
        )

        logits = self.network(x)

        return logits

In [12]:
# ============================================================
# 13. LOAD TRAINED WEIGHTS
# ============================================================

MODEL_PATH = Path(
    "./train_final/recursive_belief_updater.pt"
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE
)

loaded_config = checkpoint["model_config"]

recursive_model = BeliefUpdater(
    n_bands=loaded_config["n_freq_bins"],
    hidden_size=loaded_config["hidden_size"]
).to(DEVICE)

recursive_model.load_state_dict(
    checkpoint["model_state_dict"]
)

recursive_model.eval()

# Use the baseline saved with THIS exact model.
P_baseline = checkpoint["P_baseline"].astype(
    np.float32
)

print("✓ Recursive model loaded.")
print()
print("Model:", checkpoint["model_name"])
print(
    "Number of bands :",
    loaded_config["n_freq_bins"]
)
print(
    "Hidden dimension:",
    loaded_config["hidden_size"]
)
print(
    "Frequency range :",
    loaded_config["freq_min_mhz"],
    "to",
    loaded_config["freq_max_mhz"],
    "MHz"
)
print(
    "Frequency bin   :",
    loaded_config["freq_bin_mhz"],
    "MHz"
)
print(
    "Time bin        :",
    loaded_config["time_bin_s"],
    "s"
)
print(
    "TBPTT length    :",
    loaded_config["tbptt_length"]
)
print(
    "Baseline shape  :",
    P_baseline.shape
)

✓ Recursive model loaded.

Model: Recursive Belief Updater
Number of bands : 36
Hidden dimension: 128
Frequency range : 0 to 18000 MHz
Frequency bin   : 500 MHz
Time bin        : 0.05 s
TBPTT length    : 32
Baseline shape  : (36,)


In [13]:
# ============================================================
# 15. RECURSIVE BELIEF PROVIDER
# ============================================================

class RecursiveBeliefProvider:
    """
    Generates the online belief P_t using the trained
    recursive ML model.

    The model receives:

        previous belief
        selected action
        observed value

    and produces:

        next belief

    The provider maintains the belief state internally
    for one environment episode.
    """

    def __init__(
        self,
        model,
        baseline,
        device
    ):

        self.model = model
        self.baseline = np.asarray(
            baseline,
            dtype=np.float32
        )

        self.device = device

        self.reset()

    def reset(self):

        self.belief = torch.tensor(
            self.baseline,
            dtype=torch.float32,
            device=self.device
        ).unsqueeze(0)

    def __call__(
        self,
        t,
        action,
        observation
    ):

        # Initial belief.
        if action is None:

            return (
                self.belief
                .squeeze(0)
                .detach()
                .cpu()
                .numpy()
                .copy()
            )

        action_tensor = torch.tensor(
            [action],
            dtype=torch.long,
            device=self.device
        )

        observation_tensor = torch.tensor(
            [observation],
            dtype=torch.float32,
            device=self.device
        )

        with torch.no_grad():

            logits = self.model(
                self.belief,
                action_tensor,
                observation_tensor
            )

            self.belief = torch.sigmoid(
                logits
            )

        return (
            self.belief
            .squeeze(0)
            .detach()
            .cpu()
            .numpy()
            .copy()
        )

In [14]:
# ============================================================
# 16. CREATE RECURSIVE BELIEF PROVIDER
# ============================================================

belief_provider = RecursiveBeliefProvider(
    model=recursive_model,
    baseline=P_baseline,
    device=DEVICE,
)

print("✓ Recursive belief provider ready.")
print(
    "Initial belief shape:",
    belief_provider.belief.shape
)

✓ Recursive belief provider ready.
Initial belief shape: torch.Size([1, 36])


In [15]:
# ============================================================
# 17. CREATE SMART SCHEDULER
# ============================================================

smart_policy = SmartSchedulerPolicy(
    exploration_weight=0.10,
    recent_hit_penalty=0.25,
    hit_memory=5,
)

print("✓ SmartScheduler ready.")

✓ SmartScheduler ready.


In [16]:
# ============================================================
# 20. ONLINE INFORMATION-BARRIER CHECK
# ============================================================

def verify_online_result(result):

    actions = result["actions"]
    observations = result["observations"]
    beliefs = result["beliefs"]

    assert len(actions) == len(
        observations
    )

    assert len(actions) == len(
        beliefs
    )

    assert actions.min() >= 0
    assert actions.max() < N_FREQ_BINS

    assert np.all(
        np.isfinite(beliefs)
    )

    assert np.all(
        beliefs >= 0
    )

    assert np.all(
        beliefs <= 1
    )

    print(
        "✓ Online trajectory passed sanity checks."
    )

In [17]:
# ============================================================
# 21. RANDOM POLICY
# ============================================================

class RandomPolicy(Policy):
    """
    Uniformly selects one of the frequency bands.

    Baseline policy. It does not learn.
    """

    def __init__(
        self,
        n_bands=N_FREQ_BINS,
        seed=42
    ):

        self.n_bands = n_bands
        self.seed = seed

        self.rng = random.Random(
            self.seed
        )

    def reset(self):

        self.rng = random.Random(
            self.seed
        )

    def select_action(self, belief):

        return self.rng.randrange(
            self.n_bands
        )

    def update(
        self,
        action,
        observation,
        reward=None
    ):

        # Random policy does not learn.
        pass

In [18]:
# ============================================================
# 22. GREEDY BELIEF POLICY
# ============================================================

class GreedyPolicy(Policy):
    """
    Always scans the band with the highest
    current recursive belief.

    No exploration.
    No learning.
    """

    def __init__(
        self,
        n_bands=N_FREQ_BINS
    ):

        self.n_bands = n_bands

    def reset(self):
        pass

    def select_action(self, belief):

        belief = np.asarray(
            belief,
            dtype=np.float32
        )

        return int(
            np.argmax(belief)
        )

    def update(
        self,
        action,
        observation,
        reward=None
    ):

        # Greedy policy does not learn.
        pass

In [19]:
# ============================================================
# 23. CREATE BASELINE POLICIES
# ============================================================

random_policy = RandomPolicy(
    n_bands=N_FREQ_BINS,
    seed=42
)

greedy_policy = GreedyPolicy(
    n_bands=N_FREQ_BINS
)

smart_policy = SmartSchedulerPolicy(
    exploration_weight=0.10,
    recent_hit_penalty=0.25,
    hit_memory=5,
)

print("✓ Random policy ready.")
print("✓ Greedy policy ready.")
print("✓ Smart scheduler ready.")

✓ Random policy ready.
✓ Greedy policy ready.
✓ Smart scheduler ready.


In [20]:
# ============================================================
# 24. POLICY EVALUATION HELPER
# ============================================================

def evaluate_policy(
    env,
    policy,
    belief_provider,
    reward_function=None
):
    """
    Run one policy on one RF environment.

    The same simulator is used for every policy.
    """

    return run_policy_on_environment(
        env=env,
        policy=policy,
        belief_provider=belief_provider,
        reward_function=reward_function
    )

In [21]:
# ============================================================
# 27. EXTRACT POLICY METRICS
# ============================================================

def extract_policy_metrics(
    result
):
    """
    Convert one simulator result into a compact
    metrics dictionary.
    """

    return {
        "total_events":
            result["total_event_count"],

        "intercepted_events":
            result["intercepted_event_count"],

        "missed_events":
            len(result["missed_events"]),

        "interception_rate":
            result["event_interception_rate"],

        "mean_delay_s":
            result["mean_intercept_delay_s"],

        "total_delay_s":
            result["total_intercept_delay_s"],

        "unique_bands":
            result["unique_bands_found"],
    }

In [22]:
# ============================================================
# 18. REWARD CONFIGURATION
# ============================================================

REWARD_HIT = 1.0
REWARD_MISS = -0.05

# Small bonus for discovering a band that has never
# produced an observed hit during this episode.
REWARD_NEW_BAND = 0.25

print("Hit reward       :", REWARD_HIT)
print("Miss penalty     :", REWARD_MISS)
print("New-band bonus   :", REWARD_NEW_BAND)

Hit reward       : 1.0
Miss penalty     : -0.05
New-band bonus   : 0.25


In [23]:
# ============================================================
# 19. OBSERVABLE REWARD FUNCTION
# ============================================================

class ObservableReward:
    """
    Reward function using ONLY information available
    to the real scanner.

    It does NOT access:
        - RF ground truth outside the selected band
        - transmission-event boundaries
        - future activity
        - Oracle results

    Reward:

        miss:
            REWARD_MISS

        first observed hit on a band:
            REWARD_HIT + REWARD_NEW_BAND

        repeated observed hit:
            REWARD_HIT
    """

    def __init__(
        self,
        hit_reward=REWARD_HIT,
        miss_penalty=REWARD_MISS,
        new_band_bonus=REWARD_NEW_BAND,
    ):

        self.hit_reward = hit_reward
        self.miss_penalty = miss_penalty
        self.new_band_bonus = new_band_bonus

        self.reset()

    def reset(self):

        self.observed_hit_bands = set()

    def __call__(
        self,
        action,
        observation
    ):

        # Miss.
        if observation <= 0:

            return self.miss_penalty

        # Hit.
        reward = self.hit_reward

        # First observed hit on this band.
        if action not in self.observed_hit_bands:

            reward += self.new_band_bonus

            self.observed_hit_bands.add(
                action
            )

        return reward

In [24]:
# ============================================================
# 20. TEST REWARD FUNCTION
# ============================================================

reward_test = ObservableReward()

print(
    "Band 3, miss :",
    reward_test(3, 0)
)

print(
    "Band 3, first hit :",
    reward_test(3, 1)
)

print(
    "Band 3, repeated hit :",
    reward_test(3, 1)
)

print(
    "Band 7, first hit :",
    reward_test(7, 1)
)

Band 3, miss : -0.05
Band 3, first hit : 1.25
Band 3, repeated hit : 1.0
Band 7, first hit : 1.25


In [25]:
# ============================================================
# 21. CONTEXT BUILDER FOR LinUCB
# ============================================================

class BanditContext:

    def __init__(
        self,
        n_bands=N_FREQ_BINS
    ):

        self.n_bands = n_bands

        self.reset()

    def reset(self):

        self.scan_counts = np.zeros(
            self.n_bands,
            dtype=np.float32
        )

        self.hit_counts = np.zeros(
            self.n_bands,
            dtype=np.float32
        )

        self.age_since_scan = np.full(
            self.n_bands,
            np.inf,
            dtype=np.float32
        )

    def update(
        self,
        action,
        observation
    ):

        self.age_since_scan += 1

        self.age_since_scan[
            action
        ] = 0

        self.scan_counts[
            action
        ] += 1

        self.hit_counts[
            action
        ] += float(
            observation > 0
        )

    def get_context(
        self,
        belief,
        action
    ):

        belief = np.asarray(
            belief,
            dtype=np.float32
        )

        if belief.shape != (
            self.n_bands,
        ):
            raise ValueError(
                f"Expected belief shape "
                f"({self.n_bands},), "
                f"got {belief.shape}"
            )

        # ----------------------------------------------------
        # Candidate-band age.
        # ----------------------------------------------------

        age = self.age_since_scan[
            action
        ]

        if not np.isfinite(age):
            age_feature = 1.0
        else:
            age_feature = min(
                age / 20.0,
                1.0
            )

        # ----------------------------------------------------
        # Candidate-band observed hit rate.
        # ----------------------------------------------------

        count = self.scan_counts[
            action
        ]

        if count > 0:

            hit_rate = (
                self.hit_counts[action]
                / count
            )

        else:

            hit_rate = 0.0

        # ----------------------------------------------------
        # Candidate-band normalized scan count.
        # ----------------------------------------------------

        scan_count_feature = min(
            count / 20.0,
            1.0
        )

        context = np.concatenate([
            belief,
            np.asarray([
                age_feature,
                hit_rate,
                scan_count_feature,
            ], dtype=np.float32)
        ])

        return context.astype(
            np.float32
        )

In [26]:
# ============================================================
# 22. LINEAR CONTEXTUAL BANDIT — LinUCB
# ============================================================

class LinUCBPolicy(Policy):
    """
    Disjoint LinUCB contextual bandit.

    Each frequency band has its own linear reward model:

        reward ≈ theta_a^T x

    Action selection:

        score(a)
            =
        estimated_reward(a)
            +
        alpha * uncertainty(a)

    The uncertainty term provides exploration.

    After observing reward r:

        A_a <- A_a + x x^T

        b_a <- b_a + r x

    Therefore the learned parameters change
    after every scan.
    """

    def __init__(
        self,
        n_bands=N_FREQ_BINS,
        alpha=1.0,
        regularization=1.0,
    ):

        self.n_bands = n_bands

        self.alpha = alpha

        self.regularization = (
            regularization
        )

        self.context_builder = (
            BanditContext(
                n_bands
            )
        )

        # 36 belief features
        # + age
        # + observed hit rate
        # + scan count
        self.context_dim = (
            n_bands + 3
        )

        self.reset()

    def reset(self):

        # ----------------------------------------------------
        # One covariance matrix A_a per band.
        # ----------------------------------------------------

        self.A = [

            self.regularization
            * np.eye(
                self.context_dim,
                dtype=np.float64
            )

            for _ in range(
                self.n_bands
            )
        ]

        # ----------------------------------------------------
        # One reward vector b_a per band.
        # ----------------------------------------------------

        self.b = [

            np.zeros(
                self.context_dim,
                dtype=np.float64
            )

            for _ in range(
                self.n_bands
            )
        ]

        # Context used for the most recent action.
        self.last_context = np.zeros(
            self.context_dim,
            dtype=np.float64
        )

        self.context_builder.reset()

    def select_action(
        self,
        belief
    ):

        scores = np.empty(
            self.n_bands,
            dtype=np.float64
        )

        contexts = {}

        # ----------------------------------------------------
        # Calculate UCB score for every possible band.
        # ----------------------------------------------------

        for action in range(
            self.n_bands
        ):

            x = (
                self.context_builder
                .get_context(
                    belief,
                    action
                )
                .astype(np.float64)
            )

            contexts[action] = x

            A = self.A[action]

            b = self.b[action]

            # Estimated reward parameters:
            #
            # theta = A^-1 b
            #
            theta = np.linalg.solve(
                A,
                b
            )

            estimated_reward = (
                theta @ x
            )

            # ------------------------------------------------
            # UCB uncertainty.
            # ------------------------------------------------

            uncertainty_squared = (
                x
                @ np.linalg.solve(
                    A,
                    x
                )
            )

            uncertainty = np.sqrt(
                max(
                    uncertainty_squared,
                    0.0
                )
            )

            scores[action] = (
                estimated_reward
                +
                self.alpha
                * uncertainty
            )

        # ----------------------------------------------------
        # Select highest UCB score.
        # ----------------------------------------------------

        action = int(
            np.argmax(scores)
        )

        # ----------------------------------------------------
        # IMPORTANT:
        # Save the exact context used for this action.
        # ----------------------------------------------------

        self.last_context = (
            contexts[action]
        ).copy()

        return action

    def update(
        self,
        action,
        observation,
        reward=None
    ):

        if reward is None:

            raise ValueError(
                "LinUCB requires a reward."
            )

        x = self.last_context

        A = self.A[action]

        b = self.b[action]

        # ----------------------------------------------------
        # ACTUAL LEARNING
        # ----------------------------------------------------

        A += np.outer(
            x,
            x
        )

        b += (
            reward * x
        )

        # ----------------------------------------------------
        # Update only AFTER learning from the old context.
        # ----------------------------------------------------

        self.context_builder.update(
            action,
            observation
        )

In [27]:
# ============================================================
# 23. CREATE LinUCB POLICY
# ============================================================

linucb_policy = LinUCBPolicy(
    n_bands=N_FREQ_BINS,
    alpha=1.0,
    regularization=1.0,
)

print("✓ LinUCB contextual bandit ready.")
print(
    "Context dimension:",
    linucb_policy.context_dim
)
print(
    "Number of actions:",
    linucb_policy.n_bands
)

✓ LinUCB contextual bandit ready.
Context dimension: 39
Number of actions: 36


In [28]:
# ============================================================
# 24. CREATE REWARD FUNCTION
# ============================================================

reward_function = ObservableReward(
    hit_reward=REWARD_HIT,
    miss_penalty=REWARD_MISS,
    new_band_bonus=REWARD_NEW_BAND,
)

print("✓ Observable reward function ready.")

✓ Observable reward function ready.


In [29]:
# ============================================================
# 25. TEST LinUCB ON ONE ENVIRONMENT
# ============================================================

test_env = np.load(
    rf_files[0]
).astype(np.float32)

linucb_result = evaluate_policy(
    env=test_env,
    policy=linucb_policy,
    belief_provider=belief_provider,
    reward_function=reward_function,
)

verify_online_result(
    linucb_result
)

print("=" * 70)
print("LinUCB — SINGLE ENVIRONMENT")
print("=" * 70)

print(
    "Environment:",
    rf_files[0].name
)

print(
    "Total events:",
    linucb_result[
        "total_event_count"
    ]
)

print(
    "Intercepted:",
    linucb_result[
        "intercepted_event_count"
    ]
)

print(
    "Interception rate:",
    f"{linucb_result['event_interception_rate']:.2%}"
)

print(
    "Mean delay:",
    f"{linucb_result['mean_intercept_delay_s']:.4f}s"
)

print(
    "Total reward:",
    f"{linucb_result['rewards'].sum():.3f}"
)

print(
    "Unique bands:",
    linucb_result[
        "unique_bands_found"
    ]
)

✓ Online trajectory passed sanity checks.
LinUCB — SINGLE ENVIRONMENT
Environment: config_0.npy
Total events: 518
Intercepted: 41
Interception rate: 7.92%
Mean delay: 0.2451s
Total reward: 549.850
Unique bands: 9


In [30]:
# ============================================================
# 26. RESET BANDIT BEFORE TRAINING
# ============================================================

linucb_policy.reset()

print(
    "✓ LinUCB reset."
)

print(
    "Training will begin from a clean state."
)

✓ LinUCB reset.
Training will begin from a clean state.


In [31]:
# ============================================================
# 27. TRAIN LinUCB ACROSS ENVIRONMENTS
# ============================================================

def train_linucb(
    policy,
    environments,
    belief_provider,
    reward_function,
    verbose_every=100,
):
    """
    Train LinUCB sequentially across environments.

    LinUCB parameters persist between environments.

    The recursive ML belief model is reset for each
    environment but remains frozen.

    Only online-observable rewards are used for learning.
    """

    history = []

    total_environments = len(
        environments
    )

    # Start from a clean bandit.
    policy.reset()

    for i, path in enumerate(
        environments,
        start=1
    ):

        env = np.load(
            path
        ).astype(np.float32)

        result = run_policy_on_environment(
            env=env,
            policy=policy,
            belief_provider=belief_provider,
            reward_function=reward_function,

            # IMPORTANT:
            # Preserve LinUCB knowledge between environments.
            reset_policy=False,
        )

        mean_reward = float(
            np.mean(
                result["rewards"]
            )
        )

        total_reward = float(
            np.sum(
                result["rewards"]
            )
        )

        history.append({

            "environment":
                path.name,

            "mean_reward":
                mean_reward,

            "total_reward":
                total_reward,

            "intercepted":
                result[
                    "intercepted_event_count"
                ],

            "total_events":
                result[
                    "total_event_count"
                ],

            "interception_rate":
                result[
                    "event_interception_rate"
                ],

            "mean_delay_s":
                result[
                    "mean_intercept_delay_s"
                ],

            "unique_bands":
                result[
                    "unique_bands_found"
                ],
        })

        if (
            verbose_every
            and i % verbose_every == 0
        ):

            recent = history[
                -verbose_every:
            ]

            print(
                f"[{i:5d}/{total_environments}] "
                f"mean reward="
                f"{np.mean([x['mean_reward'] for x in recent]):.4f} | "
                f"mean ISR="
                f"{np.nanmean([x['interception_rate'] for x in recent]):.2%}"
            )

    return history

In [32]:
# ============================================================
# 28. TRAIN LinUCB — DEVELOPMENT RUN
# ============================================================

development_rf_files = rf_files[:5]

linucb_history = train_linucb(
    policy=linucb_policy,
    environments=development_rf_files,
    belief_provider=belief_provider,
    reward_function=reward_function,
    verbose_every=1,
)

print()
print(
    "✓ Development training complete."
)

[    1/5] mean reward=0.9195 | mean ISR=7.92%
[    2/5] mean reward=0.9115 | mean ISR=5.38%
[    3/5] mean reward=0.9041 | mean ISR=3.32%
[    4/5] mean reward=0.9742 | mean ISR=0.39%
[    5/5] mean reward=0.8294 | mean ISR=5.56%

✓ Development training complete.


In [33]:
# ============================================================
# 29. LOAD ORACLE METRICS
# ============================================================

ORACLE_DIR = Path(
    "./Oracle"
)

ORACLE_METRICS_PATH = (
    ORACLE_DIR
    / "oracle_metrics.json"
)

if not ORACLE_METRICS_PATH.exists():

    raise FileNotFoundError(
        "Oracle metrics were not found at "
        f"{ORACLE_METRICS_PATH}. "
        "Run Oracle.ipynb first."
    )

with open(
    ORACLE_METRICS_PATH,
    "r"
) as f:

    oracle_metrics = json.load(f)

print(
    f"✓ Loaded Oracle metrics for "
    f"{len(oracle_metrics)} environments."
)

✓ Loaded Oracle metrics for 5 environments.


In [34]:
# ============================================================
# 30. ORACLE SCHEDULE LOADER
# ============================================================

def load_oracle_schedule(
    environment_name
):

    schedule_path = (
        ORACLE_DIR
        / environment_name.replace(
            ".npy",
            "_oracle_schedule.npy"
        )
    )

    if not schedule_path.exists():

        raise FileNotFoundError(
            f"Oracle schedule not found: "
            f"{schedule_path}"
        )

    return np.load(
        schedule_path
    )

In [35]:
# ============================================================
# 31. RECONSTRUCT ORACLE EVENT INTERCEPTIONS
# ============================================================

def oracle_event_interceptions(
    env,
    oracle_schedule
):
    """
    Reconstruct event-level Oracle interceptions
    from the saved Oracle schedule.

    This does NOT rerun the Oracle optimizer.
    """

    events = extract_transmission_events(
        env
    )

    event_by_id = {
        event["id"]: event
        for event in events
    }

    intercepted_ids = set()
    interception_times = {}

    for t, band in enumerate(
        oracle_schedule
    ):

        if band < 0:
            continue

        for event_id in build_event_lookup(
            events,
            len(env)
        )[t]:

            if event_id in intercepted_ids:
                continue

            event = event_by_id[
                event_id
            ]

            if event["band"] == band:

                intercepted_ids.add(
                    event_id
                )

                interception_times[
                    event_id
                ] = t

    intercepted = {}

    for event_id, t in (
        interception_times.items()
    ):

        event = event_by_id[
            event_id
        ]

        delay_steps = (
            t - event["start"]
        )

        intercepted[event_id] = {
            **event,
            "interception_time": t,
            "delay_steps": delay_steps,
            "delay_s":
                delay_steps * TIME_BIN_S,
        }

    return intercepted

In [36]:
# ============================================================
# 32. ORACLE-RELATIVE EVALUATION
# ============================================================

def compare_to_oracle(
    environment_name,
    policy_result
):
    """
    Compare one online policy result against the
    corresponding offline Oracle result.

    Primary comparison:
        intercepted events / Oracle-intercepted events

    Delay comparison:
        only events intercepted by BOTH systems.

    This avoids comparing delay over completely
    different event sets.
    """

    env = np.load(
        TRAIN_RFENV_DIR
        / environment_name
    ).astype(np.float32)

    oracle_schedule = load_oracle_schedule(
        environment_name
    )

    oracle_events = (
        oracle_event_interceptions(
            env,
            oracle_schedule
        )
    )

    policy_events = {
        event["id"]: event
        for event in
        policy_result[
            "intercepted_events"
        ]
    }

    oracle_intercepted = len(
        oracle_events
    )

    policy_intercepted = len(
        policy_events
    )

    # --------------------------------------------------------
    # Relative interception performance.
    # --------------------------------------------------------

    oracle_relative_isr = (
        policy_intercepted
        / oracle_intercepted
        if oracle_intercepted > 0
        else np.nan
    )

    # --------------------------------------------------------
    # Events intercepted by BOTH systems.
    # --------------------------------------------------------

    common_ids = (
        set(policy_events)
        &
        set(oracle_events)
    )

    if common_ids:

        policy_common_delays = np.asarray([
            policy_events[event_id][
                "delay_s"
            ]
            for event_id
            in common_ids
        ])

        oracle_common_delays = np.asarray([
            oracle_events[event_id][
                "delay_s"
            ]
            for event_id
            in common_ids
        ])

        mean_policy_common_delay = float(
            np.mean(
                policy_common_delays
            )
        )

        mean_oracle_common_delay = float(
            np.mean(
                oracle_common_delays
            )
        )

        mean_excess_delay = float(
            np.mean(
                policy_common_delays
                - oracle_common_delays
            )
        )

    else:

        mean_policy_common_delay = np.nan
        mean_oracle_common_delay = np.nan
        mean_excess_delay = np.nan

    return {

        "environment":
            environment_name,

        "policy_intercepted":
            policy_intercepted,

        "oracle_intercepted":
            oracle_intercepted,

        "oracle_relative_isr":
            oracle_relative_isr,

        "common_events":
            len(common_ids),

        "policy_common_mean_delay_s":
            mean_policy_common_delay,

        "oracle_common_mean_delay_s":
            mean_oracle_common_delay,

        "mean_excess_delay_s":
            mean_excess_delay,
    }

In [37]:
# ============================================================
# 33. EVALUATE LinUCB AGAINST ORACLE
# ============================================================

linucb_policy.reset()

eval_result = run_policy_on_environment(
    env=test_env,
    policy=linucb_policy,
    belief_provider=belief_provider,
    reward_function=reward_function,
)

oracle_comparison = compare_to_oracle(
    environment_name=rf_files[0].name,
    policy_result=eval_result,
)

print("=" * 70)
print("LinUCB vs ORACLE")
print("=" * 70)

for key, value in oracle_comparison.items():

    print(
        f"{key:35s}: {value}"
    )

LinUCB vs ORACLE
environment                        : config_0.npy
policy_intercepted                 : 41
oracle_intercepted                 : 451
oracle_relative_isr                : 0.09090909090909091
common_events                      : 41
policy_common_mean_delay_s         : 0.24512195121951222
oracle_common_mean_delay_s         : 0.062195121951219505
mean_excess_delay_s                : 0.18292682926829268
